# Lending Club P2P Loan Default Prediction — Part 2: Improvements

Fixes three problems in the paper: data leakage, random split, and no gradient boosting.

## 1. Data and cleaning (reuse from baseline)

In [ ]:
# CELL 1 - Get the data (same as the baseline notebook)
import kagglehub, os, glob
data_dir = kagglehub.dataset_download("wordsforthewise/lending-club")
print("Data folder:", data_dir)

In [ ]:
# CELL 2 - Rebuild the paper's "data_clean.csv"
# The author's code reads a file called data_clean.csv that is NOT in the zip.
# The paper (Section III-A) says how it was made:
#   - loans issued 2012 to 2015
#   - only loans with a FINAL status: "Fully Paid", "Charged Off", "Default"
#     (so "Current", "Late", "In Grace Period" are removed)
import pandas as pd, numpy as np

csv_file = data_dir + "/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv"

# All columns the author's classification + regression code uses, plus issue_d to filter by year
author_cols = ['loan_amnt','funded_amnt','total_pymnt','term','int_rate','installment','issue_d','last_pymnt_d',
 'emp_length','home_ownership','annual_inc','verification_status','loan_status','purpose','zip_code','addr_state','dti',
 'delinq_2yrs','earliest_cr_line','fico_range_low','fico_range_high','inq_last_6mths','mths_since_last_delinq',
 'mths_since_last_record','open_acc','pub_rec','revol_bal','revol_util','total_acc','out_prncp','out_prncp_inv',
 'last_fico_range_high','last_fico_range_low','collections_12_mths_ex_med','mths_since_last_major_derog','policy_code',
 'application_type','annual_inc_joint','dti_joint','acc_now_delinq','tot_coll_amt','tot_cur_bal','open_acc_6m',
 'open_act_il','open_il_12m','open_il_24m','mths_since_rcnt_il','total_bal_il','il_util','open_rv_12m','open_rv_24m',
 'all_util','total_rev_hi_lim','inq_fi','total_cu_tl','inq_last_12m','acc_open_past_24mths','avg_cur_bal',
 'bc_open_to_buy','bc_util','chargeoff_within_12_mths','delinq_amnt','mo_sin_old_il_acct','mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op','mo_sin_rcnt_tl','mort_acc','mths_since_recent_bc','mths_since_recent_bc_dlq',
 'mths_since_recent_inq','mths_since_recent_revol_delinq','num_accts_ever_120_pd','num_actv_bc_tl','num_actv_rev_tl',
 'num_bc_sats','num_bc_tl','num_il_tl','num_op_rev_tl','num_rev_accts','num_rev_tl_bal_gt_0','num_sats',
 'num_tl_120dpd_2m','num_tl_30dpd','num_tl_90g_dpd_24m','num_tl_op_past_12m','pct_tl_nvr_dlq','percent_bc_gt_75',
 'pub_rec_bankruptcies','tax_liens','tot_hi_cred_lim','total_bal_ex_mort','total_bc_limit','total_il_high_credit_limit']

raw = pd.read_csv(csv_file, usecols=author_cols, low_memory=False)
print("Full Kaggle file:", raw.shape)

raw["issue_year"] = pd.to_datetime(raw["issue_d"], format="%b-%Y").dt.year
clean = raw[raw["issue_year"].between(2012, 2015) &
            raw["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
clean = clean.drop(columns=["issue_year"])
del raw

print("After filter (paper had 745,529):", len(clean))
print(clean["loan_status"].value_counts())
print((clean["loan_status"] != "Fully Paid").mean().round(3), "= share negative (paper: 0.19)")
clean.to_csv("data_clean.csv", index=False)

In [ ]:
# CELL 3 - Authors' cleaning rules (copied from baseline notebook)
import pandas as pd, numpy as np, gc

nan_max_cols = ['mths_since_last_delinq','mths_since_last_record','mths_since_last_major_derog','mths_since_rcnt_il',
 'mths_since_recent_bc','mths_since_recent_bc_dlq','mths_since_recent_inq','mths_since_recent_revol_delinq',
 'mo_sin_old_il_acct','mo_sin_old_rev_tl_op','mo_sin_rcnt_rev_tl_op','mo_sin_rcnt_tl']
nan_zero_cols = ['tot_coll_amt','tot_cur_bal','open_acc_6m','open_act_il','open_il_12m','open_il_24m','total_bal_il',
 'open_rv_12m','open_rv_24m','inq_fi','inq_last_12m','num_accts_ever_120_pd','num_actv_bc_tl','num_actv_rev_tl',
 'num_bc_sats','num_bc_tl','num_il_tl','num_op_rev_tl','num_rev_accts','num_rev_tl_bal_gt_0','num_sats',
 'num_tl_120dpd_2m','num_tl_30dpd','num_tl_90g_dpd_24m','num_tl_op_past_12m','mort_acc','percent_bc_gt_75',
 'tot_hi_cred_lim','total_bal_ex_mort','total_bc_limit','total_il_high_credit_limit','bc_open_to_buy',
 'acc_open_past_24mths','total_cu_tl','total_rev_hi_lim']
nan_mean_cols = ['dti','annual_inc_joint','dti_joint','il_util','all_util','bc_util','pct_tl_nvr_dlq','avg_cur_bal','revol_util']

def author_clean(df):
    df = df.copy()
    df['revol_util'] = pd.to_numeric(df['revol_util'].astype(str).str.replace('%',''), errors='coerce') / 100.0
    df['int_rate']   = pd.to_numeric(df['int_rate'].astype(str).str.replace('%',''), errors='coerce') / 100.0
    for col in df.columns:
        if col in nan_mean_cols: df[col] = df[col].fillna(np.nanmean(df[col].values))
        if col in nan_zero_cols: df[col] = df[col].fillna(0.0)
        if col in nan_max_cols:  df[col] = df[col].fillna(np.nanmax(df[col].values) * 5)
    d = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
    df['earliest_cr_line'] = (d - pd.Timestamp('1970-01-01')).dt.days
    return df

## 2. Shared setup: three settings S1/S2/S3

In [ ]:
# CELL 4 - OUR IMPROVEMENTS - shared setup
# Everything above reproduces the paper. From here we change ONE thing at a time and measure the effect.
# This is called an ABLATION STUDY: it lets us say exactly which change caused which result.
#
# Three settings (what the model is allowed to see, and how we test it):
#   S1 = paper setup:   random 70/30 split, ALL features (including the leaky ones)
#   S2 = leak removed:  random 70/30 split, leaky columns dropped
#   S3 = honest test:   leaky columns dropped AND train on 2012-2014 loans, test on 2015 loans (the future)
#
# DATA LEAKAGE = giving the model information that only exists AFTER the answer is known:
#   last_fico_range_high / low : the borrower's credit score at the END of the loan (a defaulter's score crashes)
#   out_prncp / out_prncp_inv  : principal still owed - always 0 for a finished loan, so it carries no signal
#
# Part of the training data is held out as a VALIDATION set (15%). It is used only to tune / stop training,
# never for the final score - the paper had no validation set (its own Future Work section says so).
import gc, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_fscore_support

LEAK_COLS = ['last_fico_range_high', 'last_fico_range_low', 'out_prncp', 'out_prncp_inv']

base = pd.read_csv("data_clean.csv", low_memory=False)
base['issue_year'] = pd.to_datetime(base['issue_d'], format='%b-%Y').dt.year
print("Share of defaults by issue year:")
print(base.groupby('issue_year')['loan_status'].apply(lambda s: (s != 'Fully Paid').mean()).round(3))
# Why 2015 is higher: the Kaggle file ends in Dec 2018, so a 5-year (60-month) loan from 2015 is only
# 'finished' if it was paid off early or defaulted. The 2015 test set therefore has more defaults.

def build_split(df, label_col, test_mask, val_frac=0.15, seed=0):
    # Same one-hot encoding + standardisation as the baseline, but the train/test split is GIVEN,
    # and a validation set is carved out of the training part.
    df = df.reset_index(drop=True)
    y = df[label_col].to_numpy(); test_mask = np.asarray(test_mask)
    df = df.drop(columns=[label_col])
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    num_cols = [c for c in df.columns if c not in cat_cols]
    cats = {c: sorted(df[c].dropna().unique()) for c in cat_cols}
    names = num_cols + [c + '_' + str(v) for c in cat_cols for v in cats[c]]
    X = np.zeros((len(df), len(names)), dtype=np.float32)
    X[:, :len(num_cols)] = df[num_cols].to_numpy(dtype=np.float32)
    j = len(num_cols)
    for c in cat_cols:
        codes = pd.Categorical(df[c], categories=cats[c]).codes.astype(np.int64)
        rows = np.where(codes >= 0)[0]; X[rows, j + codes[rows]] = 1.0; j += len(cats[c])
    np.nan_to_num(X, copy=False)
    tr_idx = np.where(~test_mask)[0]; te_idx = np.where(test_mask)[0]
    rng = np.random.default_rng(seed); rng.shuffle(tr_idx)
    n_val = int(val_frac * len(tr_idx)); va_idx, tr_idx = tr_idx[:n_val], tr_idx[n_val:]
    m = X[tr_idx].mean(0, dtype=np.float64).astype(np.float32); s = X[tr_idx].std(0, dtype=np.float64).astype(np.float32); s[s == 0] = 1
    X -= m; X /= s
    return dict(Xtr=X[tr_idx], ytr=y[tr_idx], Xva=X[va_idx], yva=y[va_idx], Xte=X[te_idx], yte=y[te_idx], names=names)

def split_mask(name, year):
    if name == 'S3':
        return year == 2015                                      # test = future loans
    return np.random.default_rng(0).random(len(year)) < 0.3     # test = random 30%

def classification_setting(name):
    df = base.drop(columns=['total_pymnt', 'last_pymnt_d', 'issue_d']).copy()
    df['loan_status'] = df['loan_status'].map({'Charged Off': 0, 'Default': 0, 'Fully Paid': 1})
    year = df.pop('issue_year').to_numpy()
    if name in ('S2', 'S3'):
        df = df.drop(columns=LEAK_COLS)
    return build_split(author_clean(df), 'loan_status', split_mask(name, year))

def regression_setting(name):
    df = base.dropna(subset=['last_pymnt_d']).drop(columns=['loan_status']).copy()
    days = (pd.to_datetime(df['last_pymnt_d'], format='%b-%Y') - pd.to_datetime(df['issue_d'], format='%b-%Y')).dt.days
    df['label'] = (df['total_pymnt'] / df['funded_amnt']) ** (365.0 / (days + 30))   # author's label = 1 + NAR
    year = df.pop('issue_year').to_numpy()
    df = df.drop(columns=['total_pymnt', 'issue_d', 'last_pymnt_d'])
    if name in ('S2', 'S3'):
        df = df.drop(columns=LEAK_COLS)
    return build_split(author_clean(df), 'label', split_mask(name, year))

## 3. Improvements 1-3: leak removal, time split, XGBoost

In [ ]:
# CELL 5 - Improvements 1, 2, 3: leak removal, time split, gradient boosting (classification)
# For each setting S1 / S2 / S3 we train three models:
#   Logistic Regression  - same as the paper (balanced class weights)
#   Random Forest        - same as the paper (200 trees, depth 30, 50 features per split)
#   XGBoost (GPU)        - NEW: gradient boosting, which the paper never tried. Trees are built one after another,
#                          each new tree fixing the mistakes of the trees before it. Training stops automatically
#                          when the score on the validation set stops improving (EARLY STOPPING).
#
# New metrics (added to the paper's precision / recall / F1):
#   ROC-AUC         - how well the model RANKS loans from risky to safe, over all possible cut-offs (0.5 = coin flip, 1 = perfect)
#   PR-AUC Default  - same idea, but focused on the rare Default class (baseline = share of defaults, ~0.19)
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight

ablation = []
def score(setting, model_name, y, p_paid, secs):
    pred = (p_paid >= 0.5).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y, pred, zero_division=0)
    row = dict(Setting=setting, Model=model_name, ROC_AUC=roc_auc_score(y, p_paid),
               PR_AUC_Default=average_precision_score(1 - y, 1 - p_paid),
               Default_P=p[0], Default_R=r[0], Default_F1=f[0],
               Weighted_F1=f1_score(y, pred, average='weighted'), Minutes=secs / 60)
    ablation.append(row)
    print(f"  {model_name:22s} ROC-AUC {row['ROC_AUC']:.3f} | PR-AUC(Default) {row['PR_AUC_Default']:.3f} | "
          f"Default F1 {row['Default_F1']:.2f} | Weighted F1 {row['Weighted_F1']:.2f} | {row['Minutes']:.1f} min")

def xgb_fit(d, **params):
    p = dict(n_estimators=3000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.5,
             min_child_weight=1, tree_method='hist', device='cuda', eval_metric='auc',
             early_stopping_rounds=100, random_state=0)
    p.update(params)
    m = xgb.XGBClassifier(**p)
    # balanced sample weights = the same trick the paper used for Logistic Regression (defaults count more)
    m.fit(d['Xtr'], d['ytr'], sample_weight=compute_sample_weight('balanced', d['ytr']),
          eval_set=[(d['Xva'], d['yva'])], verbose=False)
    return m

xgb_models = {}
for setting in ['S1', 'S2', 'S3']:
    d = classification_setting(setting)
    print(f"\n===== {setting}: train {len(d['ytr'])}, validation {len(d['yva'])}, test {len(d['yte'])}, features {len(d['names'])} =====")
    t = time.time(); m = LogisticRegression(class_weight='balanced', solver='lbfgs', max_iter=1000).fit(d['Xtr'], d['ytr'])
    score(setting, 'Logistic Regression', d['yte'], m.predict_proba(d['Xte'])[:, 1], time.time() - t)
    t = time.time(); m = RandomForestClassifier(n_estimators=200, n_jobs=-1, max_depth=30, max_features=50, random_state=0).fit(d['Xtr'], d['ytr'])
    score(setting, 'Random Forest', d['yte'], m.predict_proba(d['Xte'])[:, 1], time.time() - t)
    t = time.time(); m = xgb_fit(d)
    score(setting, 'XGBoost (GPU)', d['yte'], m.predict_proba(d['Xte'])[:, 1], time.time() - t)
    xgb_models[setting] = dict(model=m, names=d['names'])
    del d, m; gc.collect()

abl = pd.DataFrame(ablation)
print(abl.round(3).to_string(index=False))

## 4. Improvement 4: tune XGBoost on validation set

In [ ]:
# CELL 6 - Improvement 4: tune XGBoost on the VALIDATION set (honest setting S3 only)
# HYPERPARAMETERS are the settings we choose before training (tree depth, learning rate, ...).
# We try 12 random combinations, pick the one with the best VALIDATION ROC-AUC, and only then look at the test set once.
# (Looking at the test set while choosing would be another form of leakage.)
d = classification_setting('S3')
rng = np.random.default_rng(42)
trials = []
for i in range(12):
    params = dict(max_depth=int(rng.choice([4, 6, 8, 10])), learning_rate=float(rng.choice([0.03, 0.05, 0.1])),
                  min_child_weight=float(rng.choice([1, 5, 20])), colsample_bytree=float(rng.choice([0.3, 0.5, 0.8])),
                  subsample=float(rng.choice([0.7, 0.8, 1.0])), reg_lambda=float(rng.choice([1, 5, 10])))
    m = xgb_fit(d, **params)
    val_auc = roc_auc_score(d['yva'], m.predict_proba(d['Xva'])[:, 1])
    trials.append({**params, 'val_auc': val_auc, 'trees': m.best_iteration + 1})
    print(f"trial {i:2d}: validation ROC-AUC {val_auc:.4f}  trees {m.best_iteration + 1:4d}  {params}")

best = max(trials, key=lambda t: t['val_auc'])
best_params = {k: v for k, v in best.items() if k not in ('val_auc', 'trees')}
print('\nBest settings:', best_params)
t = time.time(); tuned = xgb_fit(d, **best_params)
score('S3', 'XGBoost tuned (GPU)', d['yte'], tuned.predict_proba(d['Xte'])[:, 1], time.time() - t)
xgb_models['S3 tuned'] = dict(model=tuned, names=d['names'])
pd.DataFrame(trials).sort_values('val_auc', ascending=False).to_csv('xgb_tuning_trials.csv', index=False)
del d; gc.collect()

## 5. Charts

In [ ]:
# CELL 7 - Two charts for the slides
# Chart 1: which features did XGBoost lean on in the PAPER'S setup (S1)? Leaky features in orange.
# Chart 2: ROC-AUC of every model in every setting - shows what each change did.
from matplotlib.patches import Patch
BLUE, ORANGE, AQUA, YELLOW = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'

imp = pd.Series(xgb_models['S1']['model'].feature_importances_, index=xgb_models['S1']['names']).sort_values(ascending=False)
top = imp.head(10)[::-1]
print('Top 10 features (share of total gain):')
print((imp.head(10) * 100).round(2).to_string())
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(top.index, top.values * 100, color=[ORANGE if n in LEAK_COLS else BLUE for n in top.index])
for i, v in enumerate(top.values * 100):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)
ax.set_title("Top 10 features XGBoost relied on in the paper's setup (S1)")
ax.set_xlabel('Feature importance (share of total gain, %)')
ax.set_ylabel('Feature')
ax.legend(handles=[Patch(color=ORANGE, label='Leaky: only known after the loan ends'), Patch(color=BLUE, label='Known when the loan is applied for')], loc='lower right')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout(); plt.savefig('chart_feature_importance_S1.png', dpi=150); plt.show()

abl = pd.DataFrame(ablation)
settings = ['S1', 'S2', 'S3']
labels = ['S1: paper setup', 'S2: leak removed', 'S3: leak removed\n+ test on future (2015)']
models = [('Logistic Regression', BLUE), ('Random Forest', ORANGE), ('XGBoost (GPU)', AQUA), ('XGBoost tuned (GPU)', YELLOW)]
fig, ax = plt.subplots(figsize=(11, 5.5))
w = 0.2
seen = set()
for k, (name, col) in enumerate(models):
    for i, s in enumerate(settings):
        row = abl[(abl.Setting == s) & (abl.Model == name)]
        if len(row):
            v = row.ROC_AUC.iloc[0]
            ax.bar(i + (k - 1.5) * w, v, w * 0.92, color=col, label=None if name in seen else name)
            seen.add(name)
            ax.text(i + (k - 1.5) * w, v + 0.01, f'{v:.3f}', ha='center', fontsize=8)
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.set_title('ROC-AUC of each model under each setting (test set)')
ax.set_xlabel('Setting')
ax.set_ylabel('ROC-AUC (0.5 = random guess, 1.0 = perfect)')
ax.axhline(0.5, color='#888888', linestyle='--', linewidth=1)
ax.legend(loc='lower left'); ax.grid(axis='y', alpha=0.25)
plt.tight_layout(); plt.savefig('chart_ablation_roc_auc.png', dpi=150); plt.show()
abl.round(4).to_csv('results_ablation_classification.csv', index=False)
print(abl.round(3).to_string(index=False))

## 6. Does the 15% strategy survive the honest setup?

In [ ]:
# CELL 8 - Does the paper's "15% a year" investment strategy survive the honest setup?
# For S1 (paper setup) and S3 (leak removed + test on 2015 loans) we train:
#   Random Forest regressor depth 10 (the paper's best; GPU, XGBoost random-forest mode as in the baseline notebook)
#   XGBoost regressor (NEW: gradient boosting on GPU, early stopping on the validation set)
# Then we rank the TEST loans by predicted return and measure the TRUE average return of the top 1.7% (the paper's
# share), top 5%, top 10%, and of ALL loans (= investing in everything, our 'no model' baseline).
import matplotlib.ticker as mtick
from sklearn.metrics import mean_squared_error, r2_score

reg_rows, curves = [], {}
for setting in ['S1', 'S3']:
    d = regression_setting(setting)
    print(f"===== {setting}: train {len(d['ytr'])}, validation {len(d['yva'])}, test {len(d['yte'])} =====")
    fitted = {
        'Random Forest depth 10': xgb.XGBRFRegressor(n_estimators=100, max_depth=10, subsample=0.632, colsample_bynode=1.0,
                                                     max_bin=128, tree_method='hist', device='cuda', random_state=0)
                                  .fit(d['Xtr'], d['ytr']),
        'XGBoost regressor': xgb.XGBRegressor(n_estimators=3000, learning_rate=0.05, max_depth=6, subsample=0.8,
                                              colsample_bytree=0.5, tree_method='hist', device='cuda',
                                              early_stopping_rounds=100, random_state=0)
                             .fit(d['Xtr'], d['ytr'], eval_set=[(d['Xva'], d['yva'])], verbose=False),
    }
    nar = d['yte'] - 1.0
    for name, m in fitted.items():
        pred = np.asarray(m.predict(d['Xte'])).ravel()
        order = np.argsort(-pred)
        row = dict(Setting=setting, Model=name, Test_MSE=mean_squared_error(d['yte'], pred), Test_R2=r2_score(d['yte'], pred))
        for share in [0.017, 0.05, 0.10]:
            k = int(share * len(nar)); row[f'Return_top_{share*100:g}%'] = nar[order[:k]].mean()
        row['Return_all_loans'] = nar.mean()
        reg_rows.append(row)
        cum = np.cumsum(nar[order]) / np.arange(1, len(nar) + 1)
        curves[(setting, name)] = (np.arange(1, len(nar) + 1)[99:] / len(nar), cum[99:])
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
    del d, fitted; gc.collect()

reg_abl = pd.DataFrame(reg_rows)
reg_abl.round(4).to_csv('results_ablation_regression.csv', index=False)
print(reg_abl.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5.5))
for (key, col, lab) in [(('S1', 'Random Forest depth 10'), BLUE, "Paper setup (S1): Random Forest"),
                        (('S3', 'Random Forest depth 10'), ORANGE, 'Honest setup (S3): Random Forest'),
                        (('S3', 'XGBoost regressor'), AQUA, 'Honest setup (S3): XGBoost')]:
    x, y = curves[key]
    ax.plot(x * 100, y * 100, color=col, linewidth=2, label=lab)
ax.axvline(1.7, color='#888888', linestyle='--', linewidth=1)
ax.text(2.2, ax.get_ylim()[1] * 0.92, "paper's 1.7% of loans", color='#444444')
ax.set_xlim(0, 50)
ax.set_title('Test set: true annualized return vs. share of loans invested (best predictions first)')
ax.set_xlabel('Percentage of loans invested')
ax.set_ylabel('Average true annualized return (NAR)')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.grid(alpha=0.25); ax.legend()
plt.tight_layout(); plt.savefig('chart_strategy_paper_vs_honest.png', dpi=150); plt.show()

## 7. Save every result in one JSON

In [ ]:
# CELL 9 - Save every result in one file (results_summary.json) for the report and slides
import json
summary = {
    "classification_ablation": abl.round(4).to_dict(orient="records"),
    "regression_ablation": reg_abl.round(4).to_dict(orient="records"),
    "top10_features_S1": (imp.head(10) * 100).round(2).to_dict(),
    "best_tuned_settings": best_params,
    "strategy_curves_sampled": {f"{s} | {m}": [[round(float(x[j]) * 100, 2), round(float(y[j]) * 100, 2)]
                                                for j in np.linspace(0, len(x) - 1, 60).astype(int)]
                                for (s, m), (x, y) in curves.items()},
}
json.dump(summary, open("results_summary.json", "w"), indent=1)
print(json.dumps(summary)[:3000])